# Bronze ingestion notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = "ipl_dataset"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
from schema.bronze import match_metadata_schema
from pyspark.sql import functions as F

json_path = "ipl_dataset/ipl_json"
df = spark.read.option("multiLine", True).json(f"/Volumes/{catalog}/zachy_ragha_raj90/{json_path}/*.json")

In [0]:
display(df.select("innings").printSchema())

In [0]:
match_df = df.select(
    F.concat(F.col("info.season"), F.lit("_"), F.col("info.event.match_number")).alias("match_id"),
    F.col("info.event.match_number").alias("match_number"),
    F.col("info.season").alias("season"),
    F.col("info.city").alias("city"),
    F.col("info.dates").alias("dates"),
    F.col("info.teams").alias("teams"),
    F.col("info.toss.winner").alias("toss_winner"),
    F.col("info.toss.decision").alias("toss_decision"),
    F.col("info.outcome.winner").alias("outcome_winner"),
    F.col("info.outcome.by.runs").alias("runs"),
    F.col("info.outcome.by.wickets").alias("wickets"),
    F.col("info.player_of_match").alias("player_of_match")
)

match_df.write.mode("overwrite").options(mergeSchema="true").format("delta").saveAsTable(f"{catalog}.zachy_ragha_raj90.ipl_matches")

In [0]:
ball_df = df.select(
    F.concat(F.col("info.season"), F.lit("_"), F.col("info.event.match_number")).alias("match_id"),
    F.explode("innings").alias("innings_split"),
    F.col("innings_split.team").alias("team"),
    F.explode("innings_split.overs").alias("overs"),
    F.col("overs.over").alias("over_id"),
    F.posexplode("overs.deliveries").alias("ball","deliveries"),
    F.struct(
        F.col("deliveries.batter").alias("batsman"),
        F.col("deliveries.bowler").alias("bowler"),
        F.col("deliveries.non_striker").alias("non_striker"),
    )

)
ball_df.select(
    F.col("match_id"),
    F.col("team"),
    F.col("over_id"),
    F.col("ball"),
    F.col("deliveries"),
).write.mode("overwrite").options(mergeSchema="true").format("delta").saveAsTable(f"{catalog}.zachy_ragha_raj90.ipl_deliveries")